# Loading Dataset

In [9]:
# ============================================================
# DAY 19 - CONTROLLED HYPERPARAMETER / PARAMETER EXPERIMENTS
# Models: LSTM, GRU, TCN
# ============================================================

import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# -----------------------------
# Training configuration
# -----------------------------
BATCH_SIZE = 32
EPOCHS = 30
BASE_LR = 0.001

print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Base learning rate:", BASE_LR)

Device: cpu
Batch size: 32
Epochs: 30
Base learning rate: 0.001


In [13]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# ============================================================
# FIND sequences_v1.npz IN CURRENT GOOGLE DRIVE
# ============================================================

import os

target = "sequences_v1.npz"
matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if target in files:
        matches.append(
            os.path.join(root, target)
        )

print("Search completed.\n")

if matches:
    print("FOUND:")
    for path in matches:
        print("✅", path)
else:
    print("❌ sequences_v1.npz was NOT found in MyDrive")

Search completed.

FOUND:
✅ /content/drive/MyDrive/IITG dataset/cleaned_dataset/processed_dataset_v1/sequences_v1.npz


In [16]:
# ============================================================
# LOAD DAY 18 DATASET
# ============================================================

import numpy as np

DATA_PATH = "/content/drive/MyDrive/IITG dataset/cleaned_dataset/processed_dataset_v1/sequences_v1.npz"

data = np.load(DATA_PATH)

print("Available arrays:")
print(data.files)

print("\nArray shapes:")
for key in data.files:
    print(key, data[key].shape)

Available arrays:
['X', 'SoH', 'RUL', 'battery_id', 'split']

Array shapes:
X (2287, 10, 2)
SoH (2287,)
RUL (2287,)
battery_id (2287,)
split (2287,)


In [17]:
# ============================================================
# DAY 19 - CHECK EXISTING DATA SPLITS
# ============================================================

X = data["X"]
SoH = data["SoH"]
RUL = data["RUL"]
battery_id = data["battery_id"]
split = data["split"]

print("X shape:", X.shape)
print("SoH shape:", SoH.shape)
print("RUL shape:", RUL.shape)
print("Battery IDs:", battery_id.shape)
print("Split:", split.shape)

print("\nUnique split values:")
print(np.unique(split, return_counts=True))

X shape: (2287, 10, 2)
SoH shape: (2287,)
RUL shape: (2287,)
Battery IDs: (2287,)
Split: (2287,)

Unique split values:
(array(['test', 'train', 'validation'], dtype='<U10'), array([ 392, 1516,  379]))


In [18]:
# ============================================================
# DAY 19 - CREATE TRAIN / VALIDATION / TEST SETS
# ============================================================

# Training data
train_idx = split == "train"

# Validation data
val_idx = split == "validation"

# Test data
test_idx = split == "test"

X_train = X[train_idx]
y_train = SoH[train_idx]

X_val = X[val_idx]
y_val = SoH[val_idx]

X_test = X[test_idx]
y_test = SoH[test_idx]

print("TRAIN")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nVALIDATION")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTEST")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

TRAIN
X_train: (1516, 10, 2)
y_train: (1516,)

VALIDATION
X_val: (379, 10, 2)
y_val: (379,)

TEST
X_test: (392, 10, 2)
y_test: (392,)


In [19]:
# ============================================================
# DAY 19 - MODEL EXPERIMENT SETUP
# ============================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import pandas as pd
import time

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [20]:
# ============================================================
# COMMON EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, X_data, y_data):

    start_time = time.time()

    predictions = model.predict(X_data, verbose=0).ravel()

    inference_time = time.time() - start_time

    mse = mean_squared_error(y_data, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_data, predictions)
    r2 = r2_score(y_data, predictions)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Inference_Time_s": inference_time
    }

In [21]:
# ============================================================
# EXPERIMENT LOG
# ============================================================

experiment_results = []

def record_experiment(
    model_name,
    experiment_name,
    config,
    training_time,
    val_metrics
):

    experiment_results.append({
        "Model": model_name,
        "Experiment": experiment_name,
        "Configuration": str(config),
        "Training_Time_s": training_time,
        "Val_MSE": val_metrics["MSE"],
        "Val_RMSE": val_metrics["RMSE"],
        "Val_MAE": val_metrics["MAE"],
        "Val_R2": val_metrics["R2"],
        "Inference_Time_s": val_metrics["Inference_Time_s"]
    })

# LSTM

In [22]:
# ============================================================
# DAY 19 - LSTM BASELINE EXPERIMENT
# ============================================================

def build_lstm_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1], X_train.shape[2])),

        layers.LSTM(units),

        layers.Dropout(dropout),

        layers.Dense(1)
    ])

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

# HYPERPARAMETER TUNING

In [23]:
# ============================================================
# DAY 19 - LSTM HYPERPARAMETER TUNING
# ============================================================

import itertools
import time
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------------------------
# Search space
# ------------------------------------------------------------
UNITS_RANGE = [32, 64, 128]
LEARNING_RATE_RANGE = [0.0001, 0.0005, 0.001]
DROPOUT_RANGE = [0.0, 0.2, 0.4]
BATCH_SIZE_RANGE = [16, 32, 64]

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

# ------------------------------------------------------------
# Generate all combinations
# ------------------------------------------------------------
lstm_combinations = list(itertools.product(
    UNITS_RANGE,
    LEARNING_RATE_RANGE,
    DROPOUT_RANGE,
    BATCH_SIZE_RANGE
))

print("Total LSTM combinations:", len(lstm_combinations))

Total LSTM combinations: 81


# GRID SEARCH

In [24]:
# ============================================================
# RUN LSTM GRID SEARCH
# ============================================================

lstm_results = []

RESULTS_FILE = "Day19_LSTM_Tuning_Results.csv"

for run_no, (units, learning_rate, dropout, batch_size) in enumerate(
    lstm_combinations, start=1
):

    print("\n" + "="*70)
    print(f"LSTM Experiment {run_no}/81")
    print(f"Units       : {units}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Dropout     : {dropout}")
    print(f"Batch Size  : {batch_size}")
    print("="*70)

    # Clear previous TensorFlow model from memory
    tf.keras.backend.clear_session()

    # Reproducibility
    tf.random.set_seed(42)
    np.random.seed(42)

    # --------------------------------------------------------
    # Build a fresh model
    # --------------------------------------------------------
    model = build_lstm_model(
        units=units,
        dropout=dropout,
        learning_rate=learning_rate
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    start_train = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0
    )

    training_time = time.time() - start_train

    # --------------------------------------------------------
    # Validation prediction + inference time
    # --------------------------------------------------------
    start_inference = time.time()

    y_val_pred = model.predict(
        X_val,
        verbose=0
    ).ravel()

    inference_time = time.time() - start_inference

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_rmse = np.sqrt(val_mse)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # --------------------------------------------------------
    # Best epoch
    # --------------------------------------------------------
    best_epoch = np.argmin(history.history["val_loss"]) + 1
    best_val_loss = np.min(history.history["val_loss"])

    # --------------------------------------------------------
    # Number of parameters
    # --------------------------------------------------------
    parameters = model.count_params()

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------
    result = {
        "Run": run_no,
        "Model": "LSTM",
        "Units": units,
        "Learning_Rate": learning_rate,
        "Dropout": dropout,
        "Batch_Size": batch_size,
        "Best_Epoch": best_epoch,
        "Best_Val_Loss": best_val_loss,
        "Val_MSE": val_mse,
        "Val_RMSE": val_rmse,
        "Val_MAE": val_mae,
        "Val_R2": val_r2,
        "Training_Time_s": training_time,
        "Inference_Time_s": inference_time,
        "Parameters": parameters
    }

    lstm_results.append(result)

    # --------------------------------------------------------
    # Save after every experiment
    # --------------------------------------------------------
    pd.DataFrame(lstm_results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"Val RMSE = {val_rmse:.6f} | "
        f"Val MAE = {val_mae:.6f} | "
        f"Val R² = {val_r2:.6f} | "
        f"Time = {training_time:.2f}s"
    )

print("\nLSTM hyperparameter tuning completed!")
print(f"Results saved to: {RESULTS_FILE}")


LSTM Experiment 1/81
Units       : 32
Learning Rate: 0.0001
Dropout     : 0.0
Batch Size  : 16
Val RMSE = 51.763985 | Val MAE = 49.171099 | Val R² = -3.582887 | Time = 22.28s

LSTM Experiment 2/81
Units       : 32
Learning Rate: 0.0001
Dropout     : 0.0
Batch Size  : 32
Val RMSE = 58.222089 | Val MAE = 54.957433 | Val R² = -4.797748 | Time = 15.08s

LSTM Experiment 3/81
Units       : 32
Learning Rate: 0.0001
Dropout     : 0.0
Batch Size  : 64
Val RMSE = 61.764593 | Val MAE = 58.116229 | Val R² = -5.524736 | Time = 9.70s

LSTM Experiment 4/81
Units       : 32
Learning Rate: 0.0001
Dropout     : 0.2
Batch Size  : 16
Val RMSE = 51.837778 | Val MAE = 49.237373 | Val R² = -3.595963 | Time = 21.05s

LSTM Experiment 5/81
Units       : 32
Learning Rate: 0.0001
Dropout     : 0.2
Batch Size  : 32
Val RMSE = 57.319433 | Val MAE = 54.152996 | Val R² = -4.619369 | Time = 13.01s

LSTM Experiment 6/81
Units       : 32
Learning Rate: 0.0001
Dropout     : 0.2
Batch Size  : 64
Val RMSE = 63.123555 | Va

# BEST LSTM CONFIGURATION

In [26]:
lstm_df = pd.DataFrame(lstm_results)

# Sort by validation RMSE
lstm_df_sorted = lstm_df.sort_values(
    by="Val_RMSE",
    ascending=True
).reset_index(drop=True)

print("TOP 10 LSTM CONFIGURATIONS")
display(
    lstm_df_sorted.head(10)
)

TOP 10 LSTM CONFIGURATIONS


,Run,Model,Units,Learning_Rate,Dropout,Batch_Size,Best_Epoch,Best_Val_Loss,Val_MSE,Val_RMSE,Val_MAE,Val_R2,Training_Time_s,Inference_Time_s,Parameters
0,68,LSTM,128,0.0005,0.2,32,28,584.665833,584.665833,24.179864,15.946983,1.962263e-05,21.208678,0.255764,67201
1,80,LSTM,128,0.0010,0.4,32,15,584.670837,584.670893,24.179969,15.947523,1.096882e-05,14.107100,0.264273,67201
2,52,LSTM,64,0.0010,0.4,16,15,584.672180,584.672200,24.179996,15.974843,8.732292e-06,11.606803,0.322927,17217
3,43,LSTM,64,0.0005,0.4,16,30,584.672302,584.672351,24.179999,15.936707,8.474610e-06,18.060938,0.248833,17217
4,81,LSTM,128,0.0010,0.4,64,27,584.675171,584.675160,24.180057,15.978644,3.669870e-06,16.510644,0.266025,67201
5,25,LSTM,32,0.0010,0.4,16,32,584.675354,584.675381,24.180062,15.919333,3.291394e-06,17.520904,0.259472,4513
6,76,LSTM,128,0.0010,0.2,16,7,584.677612,584.677585,24.180107,15.911972,-4.780371e-07,12.819535,0.273500,67201
7,49,LSTM,64,0.0010,0.2,16,15,584.678040,584.678086,24.180118,15.913301,-1.334270e-06,11.114443,0.262452,17217
8,71,LSTM,128,0.0005,0.4,32,28,584.684143,584.684170,24.180243,16.003124,-1.174106e-05,21.630302,0.269950,67201
9,47,LSTM,64,0.0010,0.0,32,29,584.687195,584.687233,24.180307,15.896811,-1.697941e-05,11.628776,0.240010,17217


# EVALUATION METRICS FOR BEST LSTM CONFIGURATION

In [27]:
best_lstm = lstm_df_sorted.iloc[0]

print("BEST LSTM CONFIGURATION")
print("-" * 40)
print(f"Units        : {best_lstm['Units']}")
print(f"Learning Rate: {best_lstm['Learning_Rate']}")
print(f"Dropout      : {best_lstm['Dropout']}")
print(f"Batch Size   : {best_lstm['Batch_Size']}")
print(f"Best Epoch   : {best_lstm['Best_Epoch']}")
print(f"Val RMSE     : {best_lstm['Val_RMSE']:.6f}")
print(f"Val MAE      : {best_lstm['Val_MAE']:.6f}")
print(f"Val R²       : {best_lstm['Val_R2']:.6f}")
print(f"Parameters   : {best_lstm['Parameters']}")
print(f"Training Time: {best_lstm['Training_Time_s']:.2f} s")

BEST LSTM CONFIGURATION
----------------------------------------
Units        : 128
Learning Rate: 0.0005
Dropout      : 0.2
Batch Size   : 32
Best Epoch   : 28
Val RMSE     : 24.179864
Val MAE      : 15.946983
Val R²       : 0.000020
Parameters   : 67201
Training Time: 21.21 s


# GRU MODEL

In [28]:
# ============================================================
# DAY 19 - GRU MODEL
# ============================================================

def build_gru_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(X_train.shape[1], X_train.shape[2])
        ),

        # GRU Layer 1
        tf.keras.layers.GRU(
            units,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(dropout),

        # GRU Layer 2
        tf.keras.layers.GRU(
            units,
            return_sequences=False
        ),

        tf.keras.layers.Dropout(dropout),

        # Fully connected layers
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [29]:
# Test GRU model
gru_test_model = build_gru_model(
    units=64,
    dropout=0.2,
    learning_rate=0.001
)

gru_test_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 10, 64)         │        13,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,129 (156.75 KB)

 Trainable params: 40,129 (156.75 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
# ============================================================
# GRU HYPERPARAMETER SEARCH SPACE
# ============================================================

gru_combinations = list(itertools.product(
    UNITS_RANGE,
    LEARNING_RATE_RANGE,
    DROPOUT_RANGE,
    BATCH_SIZE_RANGE
))

print("Total GRU combinations:", len(gru_combinations))

Total GRU combinations: 81


# GRID SEARCH

In [31]:
# ============================================================
# RUN GRU GRID SEARCH
# ============================================================

gru_results = []

RESULTS_FILE = "Day19_GRU_Tuning_Results.csv"

for run_no, (units, learning_rate, dropout, batch_size) in enumerate(
    gru_combinations, start=1
):

    print("\n" + "=" * 70)
    print(f"GRU Experiment {run_no}/81")
    print(f"Units        : {units}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Dropout      : {dropout}")
    print(f"Batch Size   : {batch_size}")
    print("=" * 70)

    # Clear previous model
    tf.keras.backend.clear_session()

    # Reproducibility
    tf.random.set_seed(42)
    np.random.seed(42)

    # --------------------------------------------------------
    # Build fresh GRU model
    # --------------------------------------------------------
    model = build_gru_model(
        units=units,
        dropout=dropout,
        learning_rate=learning_rate
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    start_train = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0
    )

    training_time = time.time() - start_train

    # --------------------------------------------------------
    # Validation inference
    # --------------------------------------------------------
    start_inference = time.time()

    y_val_pred = model.predict(
        X_val,
        verbose=0
    ).ravel()

    inference_time = time.time() - start_inference

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_rmse = np.sqrt(val_mse)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # --------------------------------------------------------
    # Best epoch
    # --------------------------------------------------------
    best_epoch = np.argmin(
        history.history["val_loss"]
    ) + 1

    best_val_loss = np.min(
        history.history["val_loss"]
    )

    # --------------------------------------------------------
    # Number of parameters
    # --------------------------------------------------------
    parameters = model.count_params()

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------
    result = {
        "Run": run_no,
        "Model": "GRU",
        "Units": units,
        "Learning_Rate": learning_rate,
        "Dropout": dropout,
        "Batch_Size": batch_size,
        "Best_Epoch": best_epoch,
        "Best_Val_Loss": best_val_loss,
        "Val_MSE": val_mse,
        "Val_RMSE": val_rmse,
        "Val_MAE": val_mae,
        "Val_R2": val_r2,
        "Training_Time_s": training_time,
        "Inference_Time_s": inference_time,
        "Parameters": parameters
    }

    gru_results.append(result)

    # --------------------------------------------------------
    # Save after every run
    # --------------------------------------------------------
    pd.DataFrame(gru_results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"Val RMSE = {val_rmse:.6f} | "
        f"Val MAE = {val_mae:.6f} | "
        f"Val R² = {val_r2:.6f} | "
        f"Time = {training_time:.2f}s"
    )

print("\nGRU hyperparameter tuning completed!")
print(f"Results saved to: {RESULTS_FILE}")


GRU Experiment 1/81
Units        : 32
Learning Rate: 0.0001
Dropout      : 0.0
Batch Size   : 16
Val RMSE = 24.182962 | Val MAE = 16.104825 | Val R² = -0.000237 | Time = 34.69s

GRU Experiment 2/81
Units        : 32
Learning Rate: 0.0001
Dropout      : 0.0
Batch Size   : 32
Val RMSE = 43.199115 | Val MAE = 41.351281 | Val R² = -2.191784 | Time = 31.53s

GRU Experiment 3/81
Units        : 32
Learning Rate: 0.0001
Dropout      : 0.0
Batch Size   : 64
Val RMSE = 47.540836 | Val MAE = 45.364214 | Val R² = -2.865604 | Time = 17.06s

GRU Experiment 4/81
Units        : 32
Learning Rate: 0.0001
Dropout      : 0.2
Batch Size   : 16
Val RMSE = 24.181071 | Val MAE = 16.039644 | Val R² = -0.000080 | Time = 38.45s

GRU Experiment 5/81
Units        : 32
Learning Rate: 0.0001
Dropout      : 0.2
Batch Size   : 32
Val RMSE = 24.564469 | Val MAE = 18.369884 | Val R² = -0.032045 | Time = 29.24s

GRU Experiment 6/81
Units        : 32
Learning Rate: 0.0001
Dropout      : 0.2
Batch Size   : 64
Val RMSE = 4

# BEST GRU

In [32]:
gru_df = pd.DataFrame(gru_results)

gru_df_sorted = gru_df.sort_values(
    by="Val_RMSE",
    ascending=True
).reset_index(drop=True)

print("TOP 10 GRU CONFIGURATIONS")

display(
    gru_df_sorted.head(10)
)

TOP 10 GRU CONFIGURATIONS


,Run,Model,Units,Learning_Rate,Dropout,Batch_Size,Best_Epoch,Best_Val_Loss,Val_MSE,Val_RMSE,Val_MAE,Val_R2,Training_Time_s,Inference_Time_s,Parameters
0,11,GRU,32,0.0005,0.0,32,10,584.669250,584.669223,24.179934,15.939704,0.000014,10.686119,0.679482,10881
1,15,GRU,32,0.0005,0.2,64,22,584.674011,584.674024,24.180034,15.934273,0.000006,12.265597,0.515027,10881
2,59,GRU,128,0.0001,0.2,32,20,584.675659,584.675679,24.180068,15.939778,0.000003,32.217725,0.524320,153921
3,29,GRU,64,0.0001,0.0,32,42,584.678040,584.678036,24.180117,15.932084,-0.000001,30.540354,0.694569,40129
4,80,GRU,128,0.0010,0.4,32,2,584.678162,584.678122,24.180118,15.937746,-0.000001,14.496915,0.539379,153921
5,60,GRU,128,0.0001,0.2,64,41,584.678772,584.678835,24.180133,15.970293,-0.000003,41.265819,0.553607,153921
6,47,GRU,64,0.0010,0.0,32,4,584.678894,584.678943,24.180135,15.969910,-0.000003,9.804193,0.500926,40129
7,68,GRU,128,0.0005,0.2,32,4,584.679871,584.679895,24.180155,15.972950,-0.000004,16.143015,0.759422,153921
8,35,GRU,64,0.0001,0.4,32,36,584.683350,584.683415,24.180228,15.917618,-0.000010,27.908165,0.503909,40129
9,37,GRU,64,0.0005,0.0,16,4,584.687195,584.687175,24.180306,15.905347,-0.000017,15.154977,0.739698,40129


# evaluation metrics

In [33]:
best_gru = gru_df_sorted.iloc[0]

print("\nBEST GRU CONFIGURATION")
print("-" * 40)
print(f"Units        : {best_gru['Units']}")
print(f"Learning Rate: {best_gru['Learning_Rate']}")
print(f"Dropout      : {best_gru['Dropout']}")
print(f"Batch Size   : {best_gru['Batch_Size']}")
print(f"Best Epoch   : {best_gru['Best_Epoch']}")
print(f"Val RMSE     : {best_gru['Val_RMSE']:.6f}")
print(f"Val MAE      : {best_gru['Val_MAE']:.6f}")
print(f"Val R²       : {best_gru['Val_R2']:.6f}")
print(f"Parameters   : {best_gru['Parameters']}")
print(f"Training Time: {best_gru['Training_Time_s']:.2f} s")


BEST GRU CONFIGURATION
----------------------------------------
Units        : 32
Learning Rate: 0.0005
Dropout      : 0.0
Batch Size   : 32
Best Epoch   : 10
Val RMSE     : 24.179934
Val MAE      : 15.939704
Val R²       : 0.000014
Parameters   : 10881
Training Time: 10.69 s


# TCN

In [34]:
def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout):

    # First causal convolution
    conv1 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    conv1 = tf.keras.layers.Dropout(dropout)(conv1)

    # Second causal convolution
    conv2 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(conv1)

    conv2 = tf.keras.layers.Dropout(dropout)(conv2)

    # Residual connection
    if x.shape[-1] != filters:
        residual = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same"
        )(x)
    else:
        residual = x

    output = tf.keras.layers.Add()([residual, conv2])
    output = tf.keras.layers.ReLU()(output)

    return output


def build_tcn_model(
    units=64,
    dropout=0.0,
    learning_rate=0.001
):

    inputs = tf.keras.layers.Input(
        shape=(X_train.shape[1], X_train.shape[2])
    )

    x = inputs

    # TCN residual blocks
    for dilation_rate in [1, 2, 4]:

        x = tcn_residual_block(
            x=x,
            filters=units,
            kernel_size=3,
            dilation_rate=dilation_rate,
            dropout=dropout
        )

    # Take final time step
    x = tf.keras.layers.Lambda(
        lambda tensor: tensor[:, -1, :]
    )(x)

    # Fully connected layers
    x = tf.keras.layers.Dense(
        32,
        activation="relu"
    )(x)

    outputs = tf.keras.layers.Dense(1)(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [35]:
# ============================================================
# TEST TCN MODEL
# ============================================================

tcn_test_model = build_tcn_model(
    units=64,
    dropout=0.2,
    learning_rate=0.001
)

tcn_test_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 10, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 10, 64)    │        448 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 10, 64)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 10, 64)    │     12,352 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 10, 64)    │        192 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 10, 64)    │          0 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 64)    │          0 │ conv1d_2[0][0],   │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 10, 64)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 10, 64)    │     12,352 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 10, 64)    │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 10, 64)    │     12,352 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 10, 64)    │          0 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 64)    │          0 │ re_lu[0][0],      │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 10, 64)    │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 10, 64)    │     12,352 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 10, 64)    │          0 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 10, 64)    │     12,352 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 10, 64)    │          0 │ conv1d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 10, 64)    │          0 │ re_lu_1[0][0],    │
│                     │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 10, 64)    │          0 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 64)        │          0 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]   

 Total params: 64,513 (252.00 KB)

 Trainable params: 64,513 (252.00 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
# ============================================================
# TCN HYPERPARAMETER SEARCH SPACE
# ============================================================

tcn_combinations = list(itertools.product(
    UNITS_RANGE,
    LEARNING_RATE_RANGE,
    DROPOUT_RANGE,
    BATCH_SIZE_RANGE
))

print("Total TCN combinations:", len(tcn_combinations))

Total TCN combinations: 81


# Grid Search

In [37]:
tcn_results = []

RESULTS_FILE = "Day19_TCN_Tuning_Results.csv"

for run_no, (units, learning_rate, dropout, batch_size) in enumerate(
    tcn_combinations, start=1
):

    print("\n" + "=" * 70)
    print(f"TCN Experiment {run_no}/81")
    print(f"Filters/Units : {units}")
    print(f"Learning Rate : {learning_rate}")
    print(f"Dropout       : {dropout}")
    print(f"Batch Size    : {batch_size}")
    print("=" * 70)

    # Clear previous TensorFlow model
    tf.keras.backend.clear_session()

    # Reproducibility
    tf.random.set_seed(42)
    np.random.seed(42)

    # --------------------------------------------------------
    # Build fresh TCN
    # --------------------------------------------------------
    model = build_tcn_model(
        units=units,
        dropout=dropout,
        learning_rate=learning_rate
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    start_train = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=0
    )

    training_time = time.time() - start_train

    # --------------------------------------------------------
    # Validation inference
    # --------------------------------------------------------
    start_inference = time.time()

    y_val_pred = model.predict(
        X_val,
        verbose=0
    ).ravel()

    inference_time = time.time() - start_inference

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_rmse = np.sqrt(val_mse)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # --------------------------------------------------------
    # Best epoch
    # --------------------------------------------------------
    best_epoch = np.argmin(
        history.history["val_loss"]
    ) + 1

    best_val_loss = np.min(
        history.history["val_loss"]
    )

    # --------------------------------------------------------
    # Number of parameters
    # --------------------------------------------------------
    parameters = model.count_params()

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------
    result = {
        "Run": run_no,
        "Model": "TCN",
        "Units": units,
        "Learning_Rate": learning_rate,
        "Dropout": dropout,
        "Batch_Size": batch_size,
        "Best_Epoch": best_epoch,
        "Best_Val_Loss": best_val_loss,
        "Val_MSE": val_mse,
        "Val_RMSE": val_rmse,
        "Val_MAE": val_mae,
        "Val_R2": val_r2,
        "Training_Time_s": training_time,
        "Inference_Time_s": inference_time,
        "Parameters": parameters
    }

    tcn_results.append(result)

    # --------------------------------------------------------
    # Save after every run
    # --------------------------------------------------------
    pd.DataFrame(tcn_results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"Val RMSE = {val_rmse:.6f} | "
        f"Val MAE = {val_mae:.6f} | "
        f"Val R² = {val_r2:.6f} | "
        f"Time = {training_time:.2f}s"
    )

print("\nTCN hyperparameter tuning completed!")
print(f"Results saved to: {RESULTS_FILE}")


TCN Experiment 1/81
Filters/Units : 32
Learning Rate : 0.0001
Dropout       : 0.0
Batch Size    : 16
Val RMSE = 22.738900 | Val MAE = 15.863602 | Val R² = 0.115653 | Time = 12.24s

TCN Experiment 2/81
Filters/Units : 32
Learning Rate : 0.0001
Dropout       : 0.0
Batch Size    : 32
Val RMSE = 24.570709 | Val MAE = 20.260512 | Val R² = -0.032569 | Time = 6.82s

TCN Experiment 3/81
Filters/Units : 32
Learning Rate : 0.0001
Dropout       : 0.0
Batch Size    : 64
Val RMSE = 27.631304 | Val MAE = 24.985316 | Val R² = -0.305830 | Time = 7.39s

TCN Experiment 4/81
Filters/Units : 32
Learning Rate : 0.0001
Dropout       : 0.2
Batch Size    : 16
Val RMSE = 22.301500 | Val MAE = 15.611780 | Val R² = 0.149348 | Time = 17.18s

TCN Experiment 5/81
Filters/Units : 32
Learning Rate : 0.0001
Dropout       : 0.2
Batch Size    : 32
Val RMSE = 24.152454 | Val MAE = 18.005544 | Val R² = 0.002285 | Time = 7.99s

TCN Experiment 6/81
Filters/Units : 32
Learning Rate : 0.0001
Dropout       : 0.2
Batch Size   

# Best TCN Configuration

In [38]:
# ============================================================
# BEST TCN CONFIGURATION
# ============================================================

tcn_df = pd.DataFrame(tcn_results)

tcn_df_sorted = tcn_df.sort_values(
    by="Val_RMSE",
    ascending=True
).reset_index(drop=True)

print("TOP 10 TCN CONFIGURATIONS")

display(
    tcn_df_sorted.head(10)
)

TOP 10 TCN CONFIGURATIONS


,Run,Model,Units,Learning_Rate,Dropout,Batch_Size,Best_Epoch,Best_Val_Loss,Val_MSE,Val_RMSE,Val_MAE,Val_R2,Training_Time_s,Inference_Time_s,Parameters
0,62,TCN,128,0.0001,0.4,32,6,478.071777,478.071784,21.864853,14.768245,0.182332,16.592168,0.667237,251841
1,63,TCN,128,0.0001,0.4,64,11,479.154724,479.154701,21.889603,14.908522,0.180480,19.257426,0.463020,251841
2,72,TCN,128,0.0005,0.4,64,4,483.076508,483.076522,21.979002,15.041747,0.173772,13.497105,0.365654,251841
3,52,TCN,64,0.0010,0.4,16,1,483.735443,483.735421,21.993986,14.523405,0.172645,8.419625,0.336820,64513
4,32,TCN,64,0.0001,0.2,32,10,485.789581,485.789540,22.040634,14.700004,0.169132,11.466639,0.338460,64513
5,54,TCN,64,0.0010,0.4,64,4,485.798981,485.799005,22.040849,14.733091,0.169116,7.555254,0.347368,64513
6,61,TCN,128,0.0001,0.4,16,4,486.460327,486.460342,22.055846,14.858948,0.167985,17.647916,0.372209,251841
7,31,TCN,64,0.0001,0.2,16,7,486.921753,486.921725,22.066303,14.803900,0.167196,12.251559,0.350331,64513
8,40,TCN,64,0.0005,0.2,16,2,487.112457,487.112463,22.070624,14.000276,0.166870,9.035888,0.323430,64513
9,36,TCN,64,0.0001,0.4,64,16,488.002625,488.002651,22.090782,15.138770,0.165347,12.243772,0.336829,64513


# Evaluation Metrics

In [39]:
best_tcn = tcn_df_sorted.iloc[0]

print("\nBEST TCN CONFIGURATION")
print("-" * 40)
print(f"Filters/Units : {best_tcn['Units']}")
print(f"Learning Rate : {best_tcn['Learning_Rate']}")
print(f"Dropout       : {best_tcn['Dropout']}")
print(f"Batch Size    : {best_tcn['Batch_Size']}")
print(f"Best Epoch    : {best_tcn['Best_Epoch']}")
print(f"Val RMSE      : {best_tcn['Val_RMSE']:.6f}")
print(f"Val MAE       : {best_tcn['Val_MAE']:.6f}")
print(f"Val R²        : {best_tcn['Val_R2']:.6f}")
print(f"Parameters    : {best_tcn['Parameters']}")
print(f"Training Time : {best_tcn['Training_Time_s']:.2f} s")


BEST TCN CONFIGURATION
----------------------------------------
Filters/Units : 128
Learning Rate : 0.0001
Dropout       : 0.4
Batch Size    : 32
Best Epoch    : 6
Val RMSE      : 21.864853
Val MAE       : 14.768245
Val R²        : 0.182332
Parameters    : 251841
Training Time : 16.59 s
